# Prioritising epitopes in one antigen, end to end

The decision this notebook walks through: **given NY-ESO-1 and a four-allele HLA panel, which peptides do we synthesise first?**

NY-ESO-1 (CTAG1B, UniProt [P78358](https://rest.uniprot.org/uniprotkb/P78358.fasta)) is a cancer-testis antigen with a well-described immunodominant HLA-A\*02:01 epitope at residues 157–165, `SLLMWITQC`. That known answer makes it a good test case: a pipeline that cannot find it is broken.

Switch `PREDICTOR` to `"mhcflurry"` if you have installed the extra and fetched its models. With `"toy"` the numbers are a deterministic placeholder — fine for following the mechanics, not for making decisions.

In [ ]:
from pathlib import Path

from epitope_rank.fasta import read_fasta
from epitope_rank.predictors import get_predictor
from epitope_rank.ranking import RankingConfig, rank_peptides
from epitope_rank.tiling import tile

PREDICTOR = "toy"  # or "mhcflurry"
ALLELES = ["HLA-A*02:01", "HLA-A*01:01", "HLA-B*07:02", "HLA-B*08:01"]
FASTA = Path("..") / "data" / "NY-ESO-1_P78358.fasta"

antigen = read_fasta(FASTA)[0]
print(antigen.name)
print(f"{len(antigen)} aa")
print(f"residues 157-165: {antigen.sequence[156:165]}")

## 1. Tile

Every 9mer and 10mer, each carrying its 1-based start position so a hit can be traced back to the antigen.

In [ ]:
peptides = tile(antigen.sequence, lengths=[9, 10], protein="NY-ESO-1")
print(f"{len(peptides)} peptides")
for p in peptides[:3]:
    print(f"  {p.start:>3}-{p.end:<3} {p.sequence}")

## 2. Score and rank

Each distinct peptide is scored once per allele, then collapsed to one row per position: its best allele, and how many alleles in the panel bind it.

Breadth leads the sort. A peptide binding all four alleles covers more of the patient population per peptide made, and synthesis is the budget.

In [ ]:
predictor = get_predictor(PREDICTOR)
shortlist, all_scores = rank_peptides(
    peptides, ALLELES, predictor, RankingConfig(top_n=10)
)

print(f"{len(all_scores)} peptide x allele pairs scored")
print(f"{int(all_scores['is_binder'].sum())} pairs cleared both cutoffs\n")
shortlist[["rank", "peptide", "start", "best_allele", "best_percentile", "n_alleles_bound"]]

## 3. Did it find the known epitope?

The check that matters: does the shortlist overlap 157–165?

In [ ]:
KNOWN_START, KNOWN_END = 157, 165

overlapping = shortlist[
    (shortlist["start"] <= KNOWN_END) & (shortlist["end"] >= KNOWN_START)
]
print(f"{len(overlapping)} of {len(shortlist)} shortlisted peptides overlap SLLMWITQC (157-165)")
overlapping[["rank", "peptide", "start", "end", "n_alleles_bound"]]

## 4. Explain a rejection

`all_scores` is kept so that "why isn't this peptide on the list?" has an answer with numbers in it. Here is the worst-scoring peptide and what each allele did with it.

In [ ]:
worst = all_scores.sort_values("percentile", ascending=False).iloc[0]["peptide"]
print(f"rejected: {worst}")
all_scores[all_scores["peptide"] == worst][
    ["peptide", "allele", "affinity_nm", "percentile", "is_binder"]
]

## 5. Look at the whole antigen

Strong binders at the top, shortlist circled, position on the x axis. The useful question of this plot is whether hits cluster into a region worth tiling more finely, or scatter.

In [ ]:
from IPython.display import Image

from epitope_rank.plotting import plot_landscape

png = plot_landscape(
    all_scores,
    shortlist,
    "results/landscape.png",
    protein_length=len(antigen),
    title=f"NY-ESO-1 ({predictor.name})",
)
Image(filename=str(png))

## What this does not tell you

Predicted binding is one necessary condition out of several. Processing, transport, the available T-cell repertoire and tolerance all sit between a tight binder and an actual response, and none of them are modelled here. The next steps a real campaign would add: filter candidates against the human proteome for cross-reactivity, weight breadth by allele frequency in the population being treated, and send the shortlist out for synthesis and an immunogenicity readout — which is the point where this list stops being a prediction and starts being data.